# 01 — Rapport global AP-HP, étape par étape

Ce notebook **reproduit chaque section** de la page produite par
`report_builder.build_rapport_global`, avec les **graphiques affichés inline**
(un graphe = une sortie de cellule), pour explorer le rapport section par section.

Les cellules appellent les **mêmes fonctions `chart_utils`** que le builder, sur les
données filtrées de la même façon. La vue page complète (IFrame) est conservée à la fin
comme référence.

> ⚠ Cette approche **duplique** volontairement la logique de `build_rapport_global`.
> Si `report_builder` évolue, re-synchroniser ce notebook.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from pathlib import Path
import pandas as pd
from IPython.display import HTML

DATA_DIR   = Path('../data')
OUTPUT_DIR = Path('../output')
OUTPUT_DIR.mkdir(exist_ok=True)

# Mode RÉEL : aligne le bandeau des pages embarquées (pas de « Données simulées »).
from report_builder import set_fake_data
set_fake_data(False)

# On charge via les loaders du builder (ajout des lignes TOTAL appareil/organe, normalisation)
from report_builder import (
    load_aphp, load_regional, load_survival,
    appareil_counts_table, survival_delay_table,
)
from chart_utils import (
    line_evolution, bar_comparison, donut_market_share, waterfall_trends,
    regional_comparison, bar_appareils_years,
    TREATMENT_COLS, GHU_LIST, REGIONAL_COLORS,
)

aphp = load_aphp(DATA_DIR)
reg  = load_regional(DATA_DIR)
surv = load_survival(DATA_DIR)

# Sous-ensembles utiles (identiques à build_rapport_global)
aphp_total = aphp[(aphp.entite == 'AP-HP') & (aphp.appareil == 'TOTAL')]
ghu_total  = aphp[(aphp.entite.isin(GHU_LIST)) & (aphp.appareil == 'TOTAL')]

years = sorted(aphp_total.annee.unique())
last_year, prev_year = years[-1], years[-2]
lv = aphp_total[aphp_total.annee == last_year].iloc[0]
pv = aphp_total[aphp_total.annee == prev_year].iloc[0]
print(f'Annees {years} - derniere={last_year}, precedente={prev_year}')

## Section 1 — Indicateurs clés

La page affiche 6 cartes KPI (dernière année vs N-1). Vue tableau équivalente :

In [ ]:
mesures = [
    ('Patients (total)',        'nb_patients'),
    ('Nouveaux patients',       'nb_nouveaux_patients'),
    ('Sejours chirurgie',       'nb_sejours_chirurgie'),
    ('Sejours chimiotherapie',  'nb_sejours_chimiotherapie'),
    ('Sejours radiotherapie',   'nb_sejours_radiotherapie'),
    ('Soins palliatifs',        'nb_sejours_palliatifs'),
]
pd.DataFrame([
    {'Indicateur': lab, str(prev_year): int(pv[col]), str(last_year): int(lv[col]),
     'Var. %': round((lv[col]-pv[col])/pv[col]*100, 1) if pv[col] else None}
    for lab, col in mesures
])

## Section 2 — Contexte régional

`regional_comparison` (évolution AP-HP vs types d'établissement) +
`donut_market_share` (répartition par type, dernière année AP-HP).

In [ ]:
reg_total = reg[(reg.appareil == 'TOTAL')]
fig_reg_pts = regional_comparison(reg_total, 'nb_patients',
                                  'Patients - AP-HP vs contexte regional',
                                  color_map=REGIONAL_COLORS)
fig_reg_pts.show()

In [ ]:
reg_total_last = reg[(reg.appareil == 'TOTAL') & (reg.organe == 'TOTAL')
                     & (reg.annee == last_year)]
types_etab = sorted(reg_total_last['entite'].unique())
fig_reg_donut = donut_market_share(reg_total_last, 'entite', 'nb_patients',
                                   f'Repartition par type d etablissement - {last_year}',
                                   entities=types_etab, color_map=REGIONAL_COLORS)
fig_reg_donut.show()

## Section 3 — Évolution globale

`waterfall_trends` (variation annuelle) + `line_evolution` des séjours par mode de PEC.

In [ ]:
fig_wf = waterfall_trends(aphp_total, 'Variation annuelle du nombre de patients - AP-HP')
fig_wf.show()

In [ ]:
melted = aphp_total.melt(id_vars=['annee'], value_vars=list(TREATMENT_COLS.keys()),
                         var_name='type_sejour', value_name='nb_sejours')
melted['label'] = melted['type_sejour'].map(TREATMENT_COLS)
fig_sejours = line_evolution(melted, 'annee', 'nb_sejours', 'label',
                             'Evolution des sejours par mode de prise en charge',
                             entities=list(TREATMENT_COLS.values()), y_zero=True)
fig_sejours.show()

## Section 4 — Répartition entre les groupes hospitaliers (GHU)

`donut_market_share` (dernière année) + `bar_comparison` (évolution groupée).

In [ ]:
ghu_last = ghu_total[ghu_total.annee == last_year]
fig_donut = donut_market_share(ghu_last, 'entite', 'nb_patients',
                               f'Repartition par GHU - {last_year}')
fig_donut.show()

In [ ]:
fig_ghu_bar = bar_comparison(ghu_total, 'annee', 'nb_patients', 'entite',
                             'Patients par GHU - evolution', barmode='group',
                             entities=GHU_LIST)
fig_ghu_bar.show()

## Section 5 — Analyse par appareil

`bar_appareils_years` (patients par appareil et par année) + tableau chiffré
`appareil_counts_table` (affiché en HTML, comme dans la page).

In [ ]:
fig_bar_app = bar_appareils_years(aphp)
fig_bar_app.show()

In [ ]:
HTML(appareil_counts_table(aphp, 'AP-HP'))

## Section 6 — Survie et délais de prise en charge (par appareil)

Tableau `survival_delay_table` (survie à 5 ans pondérée + délai médian), affiché en HTML.

In [ ]:
HTML(survival_delay_table(surv, aphp, 'AP-HP'))

## Page complète (référence)

Génération du HTML complet via `build_rapport_global` et affichage en IFrame.

In [ ]:
from report_builder import build_rapport_global

out = build_rapport_global(DATA_DIR, OUTPUT_DIR)
print(f'Rapport genere : {out}')

from IPython.display import IFrame
IFrame(os.path.relpath(out), width='100%', height=800)